# Notebook 06: Mixed Precision Training (AMP)

**Module:** 11 Production ML  
**Duration:** ~2 hrs

---

## Learning Objectives

1. Understand FP16 vs FP32 training
2. Use torch.cuda.amp.autocast and GradScaler
3. Know when AMP helps GeoSpatial training
4. Avoid numerical instability

## Automatic Mixed Precision

**FP16:** Half precision — 2× faster, half memory on Tensor Cores.

**GradScaler:** Prevents gradient underflow in FP16.

```python
scaler = torch.cuda.amp.GradScaler()
with torch.cuda.amp.autocast():
    loss = model(x, y)
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
```

In [ ]:
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
scaler = torch.amp.GradScaler('cuda' if torch.cuda.is_available() else 'cpu')
model = nn.Linear(64, 2).to(device)
opt = torch.optim.Adam(model.parameters())
x = torch.randn(8, 64, device=device)
y = torch.randint(0, 2, (8,), device=device)

for _ in range(3):
    opt.zero_grad()
    with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
        loss = nn.CrossEntropyLoss()(model(x), y)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
print(f'AMP training step OK, loss={loss.item():.4f}')

---

## Summary

AMP speeds training ~1.5–2× on modern GPUs with minimal accuracy loss.

**Next:** [07_Distributed_Training.ipynb](07_Distributed_Training.ipynb)